# Scrape New SF Court Cases (Local)

Find new small claims cases by **probing case-number ranges** and add them to
`valid_cases.json`. **Run this on your laptop, not in Colab** — Cloudflare
blocks Colab IPs.

### What this does
1. Loads existing `valid_cases.json` so we don't re-probe known cases
2. Asks for a SessionID (manual — Cloudflare blocks automation here)
3. Starts a background keepalive thread to help extend the session
4. Probes every case number in the configured range via `GetROA`
5. Saves valid cases to `valid_cases.json` incrementally — interrupt-safe

### Why probing instead of dates?
The court calendar API (`GetCases2`) only returns recent dates — anything more
than a couple months old comes back empty. The `GetROA` endpoint works for any
case that exists, regardless of age, so we brute-force a numeric range of
case numbers instead. The case-number prefix encodes year: `CSM25...` = 2025,
`CSM26...` = 2026.

In [ ]:
import json
import time
from pathlib import Path

# ============================================================
# CONFIGURATION
# ============================================================
PROBE_PREFIX    = "CSM"           # case number prefix (CSM = small claims)
PROBE_START_NUM = 26871856        # inclusive — Phase C: extend 2026 above prior probed max (26871855)
PROBE_END_NUM   = 26873000        # inclusive — stop here unless hit rate stays high
PROBE_STEP      = 1               # dense probe — collect every existing case
PROBE_DELAY     = 1.0             # seconds between probes
# ============================================================

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VALID_CASES_PATH = REPO_ROOT / "scraper" / "state" / "valid_cases.json"

BASE_URL = "https://webapps.sftc.org"
CALENDAR_PATH = "/cc/CaseCalendar.dll"  # used by keepalive
CASE_PATH     = "/ci/CaseInfo.dll"      # used by probing
COURT_URL = f"{BASE_URL}{CALENDAR_PATH}"
CASE_TYPE = "M//CSM"                    # used by keepalive
USER_AGENT = "MSDS603-Research-Scraper/1.0 (SF Small Claims academic study)"

total_in_range = len(range(PROBE_START_NUM, PROBE_END_NUM + 1, PROBE_STEP))
print(f"Repo root:        {REPO_ROOT}")
print(f"valid_cases.json: {VALID_CASES_PATH}")
print(f"Probe range:      {PROBE_PREFIX}{PROBE_START_NUM} → {PROBE_PREFIX}{PROBE_END_NUM} (step {PROBE_STEP})")
print(f"  ({total_in_range} candidates)")

## Step 1 — Load existing valid_cases.json

Reads the cases we've already discovered so we can skip them. Anything already
in `probed` or `valid` will NOT be re-fetched.

In [ ]:
if VALID_CASES_PATH.exists():
    store = json.loads(VALID_CASES_PATH.read_text())
    existing_cases = set(store.get("valid", {}).keys())
    already_probed = set(store.get("probed", []))
    print(f"✓ Loaded {VALID_CASES_PATH.name}")
    print(f"  Already-known valid cases: {len(existing_cases)}")
    print(f"  Already-probed (any result): {len(already_probed)}")
else:
    store = {"valid": {}, "not_found": [], "probed": []}
    existing_cases = set()
    already_probed = set()
    print(f"⚠ {VALID_CASES_PATH.name} does not exist — starting fresh")

## Step 2 — Get a SessionID and start the keepalive

The court site is gated by Cloudflare's "verify you're a person" check, so the
very first step needs a human. You solve the verification once in your browser
and paste the SessionID here.

After that, a background thread pings the court API every 60 seconds. Note: the
site likely uses a hard session TTL (not idle), so pings may not fully prevent
expiry — the probing loop in Step 3 handles re-prompting gracefully if needed.

**To get a SessionID:**
1. Open https://webapps.sftc.org/cc/CaseCalendar.dll in your browser
2. Solve the Cloudflare verification (one click)
3. Once the calendar loads, copy the hex string after `SessionID=` in the URL bar

In [ ]:
import threading
import requests
from datetime import date as _date

# Mutable holder so the keepalive thread can pick up new sessions if expiry happens
_session_holder = {"id": ""}
_keepalive_started = {"flag": False}


def start_keepalive(session_id: str, ping_interval: int = 60) -> None:
    """Start (or update) a daemon thread that pings the court API every
    `ping_interval` seconds. Idempotent — calling again with a new session_id
    just updates the holder; only one thread ever runs per kernel."""
    _session_holder["id"] = session_id
    if _keepalive_started["flag"]:
        print("  ↻ Keepalive updated with new SessionID")
        return
    _keepalive_started["flag"] = True

    def ping_loop():
        while True:
            sid = _session_holder["id"]
            if sid:
                try:
                    requests.get(
                        f"{BASE_URL}{CALENDAR_PATH}/datasnap/rest/TServerMethods1/"
                        f"GetCases2/{_date.today().isoformat()}/{CASE_TYPE}/{sid}",
                        headers={"User-Agent": USER_AGENT},
                        timeout=10,
                    )
                except Exception:
                    pass
            time.sleep(ping_interval)

    threading.Thread(target=ping_loop, daemon=True).start()
    print(f"  ✓ Keepalive thread started (pinging every {ping_interval}s)")


print("=" * 60)
print("SESSION ID NEEDED")
print("=" * 60)
print(f"\n1. Open this URL in your browser:")
print(f"   {COURT_URL}")
print(f"\n2. Solve the Cloudflare verification (click the checkbox)")
print(f"\n3. After the calendar loads, copy the hex string after `SessionID=`")
print(f"   in the URL bar. Example:")
print(f"     https://...?SessionID=A1B2C3D4...  →  copy A1B2C3D4...")
print()

SESSION_ID = input("Paste SessionID: ").strip()
print(f"\n✓ Using SessionID: {SESSION_ID[:12]}... ({len(SESSION_ID)} chars)")

start_keepalive(SESSION_ID)

## Step 3 — Probe case numbers in the configured range

For each case number in `PROBE_START_NUM` to `PROBE_END_NUM`, hits:
```
GET /ci/CaseInfo.dll/datasnap/rest/TServerMethods1/GetROA/{case_num}/{SessionID}/
```

The response is `{"result": [count, "[{...ROA entries...}]"]}`:
- `count == -1` → session expired (loop refreshes and retries the same case)
- `count == 0` → case doesn't exist (mark probed, move on)
- otherwise → case exists, save to `valid_cases.json`

Already-probed cases (from previous runs) are skipped automatically.
Progress is saved every 25 probes — safe to interrupt at any time.

In [ ]:
def probe_case(case_num: str, sid: str):
    """Probe one case via GetROA. Returns (status, doc_count).
    status: 'expired' | 'found' | 'not_found' | 'error'"""
    url = f"{BASE_URL}{CASE_PATH}/datasnap/rest/TServerMethods1/GetROA/{case_num}/{sid}/"
    try:
        resp = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        code = data["result"][0]
        if code == -1:
            return "expired", 0
        if code == 0:
            return "not_found", 0
        roa = json.loads(data["result"][1])
        return "found", len(roa)
    except Exception:
        return "error", 0


def save_progress() -> None:
    """Atomic-ish write of the in-memory store to disk."""
    VALID_CASES_PATH.parent.mkdir(parents=True, exist_ok=True)
    VALID_CASES_PATH.write_text(json.dumps(store, indent=2))


# Build target list, skipping already-probed cases (resume support)
candidates = [
    f"{PROBE_PREFIX}{n}"
    for n in range(PROBE_START_NUM, PROBE_END_NUM + 1, PROBE_STEP)
    if f"{PROBE_PREFIX}{n}" not in already_probed
]

print(f"Range:           {PROBE_PREFIX}{PROBE_START_NUM} → {PROBE_PREFIX}{PROBE_END_NUM} (step {PROBE_STEP})")
print(f"  Total in range:  {total_in_range}")
print(f"  Already probed:  {total_in_range - len(candidates)}")
print(f"  To probe:        {len(candidates)}")
print(f"  At {PROBE_DELAY}s/probe, expect ~{len(candidates) * PROBE_DELAY / 60:.1f} min")
print(f"  Saved every 25 probes — safe to interrupt.\n")

probed = 0
found = 0
not_found = 0
errors = 0
SAVE_EVERY = 25

for i, case_num in enumerate(candidates, 1):
    while True:  # retry loop for session expiry
        time.sleep(PROBE_DELAY)
        status, doc_count = probe_case(case_num, SESSION_ID)

        if status == "expired":
            save_progress()
            print(f"  [{i}/{len(candidates)}] {case_num}: session expired (saved {found} found so far)")
            print("  → Get a fresh SessionID from your browser (refresh the court URL)")
            SESSION_ID = input("  Paste new SessionID: ").strip()
            start_keepalive(SESSION_ID)
            continue  # retry same case

        if status == "found":
            found += 1
            store.setdefault("valid", {})[case_num] = doc_count
            if case_num not in store.get("probed", []):
                store.setdefault("probed", []).append(case_num)
            already_probed.add(case_num)
            print(f"  [{i}/{len(candidates)}] {case_num}: FOUND ({doc_count} ROA entries)")
        elif status == "not_found":
            not_found += 1
            if case_num not in store.get("not_found", []):
                store.setdefault("not_found", []).append(case_num)
            if case_num not in store.get("probed", []):
                store.setdefault("probed", []).append(case_num)
            already_probed.add(case_num)
            print(f"  [{i}/{len(candidates)}] {case_num}: not found")
        else:
            errors += 1
            print(f"  [{i}/{len(candidates)}] {case_num}: ERROR")

        probed += 1
        break

    if i % SAVE_EVERY == 0:
        save_progress()

save_progress()

print("\n" + "=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  Probed this run:  {probed}")
print(f"  Found (NEW):      {found}")
print(f"  Not found:        {not_found}")
print(f"  Errors:           {errors}")
print(f"  Total valid now:  {len(store.get('valid', {}))}")
print(f"  Saved to:         {VALID_CASES_PATH}")